¿Cuáles son los errores de posicionamiento más frecuentes que presentan los ciclistas antes de un estudio biomecánico y qué ajustes son los que corrigen con mayor frecuencia esos problemas?

# PARTE 1. EXTRACCIÓN Y LECTURA DE DATOS

## 1. Importación de librerías

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import getpass
import mysql.connector



## 2. Definición de rutas

In [2]:
RUTA_PROYECTO = Path(
    r"C:\Users\miren\OneDrive\Datos adjuntos\Documentos\Especialidad IT ACADEMY\sprint 13 Proyecto final"
)

RUTA_RAW = RUTA_PROYECTO / "data" / "raw"
RUTA_PROCESSED = RUTA_PROYECTO / "data" / "processed"



## 3. Inventario de archivos

In [3]:
archivos = sorted(
    archivo
    for archivo in RUTA_RAW.iterdir()
    if archivo.is_file() and archivo.suffix.lower() != ".zip"
)

print(f"Número de archivos de datos encontrados: {len(archivos)}")

for archivo in archivos:
    print(archivo.name)

Número de archivos de datos encontrados: 11
1758_Right.pose
1759_Right.pose
1800_Left.pose
1801_Left.pose
1804_Left.pose
1805_Right.pose
info.bike
info.client
info.session
medidas_finales.zin
medidas_iniciales.zin


## 4. Lectura de los datos del ciclista


In [4]:
RUTA_CLIENT = RUTA_RAW / "info.client"

raiz_client = ET.parse(RUTA_CLIENT).getroot()

df_ciclista = pd.DataFrame([{elemento.tag: elemento.text for elemento in raiz_client}])
df_ciclista.insert(0, "cyclist_id", "CYC_001")

df_ciclista

,cyclist_id,lastname,firstname,suffix,phone,e-mail,gender,birthday,ridingstyle,goals,injuries,training,flexibility
0,CYC_001,Apellido,Nombre,None,123456789,correo@ejemplo.com,Male,1990-01-01,Elite,None,None,None,High


## 5. Lectura de los datos de la bicicleta


In [5]:
RUTA_BIKE = RUTA_RAW / "info.bike"

raiz_bike = ET.parse(RUTA_BIKE).getroot()

df_bicicleta = pd.DataFrame([{elemento.tag: elemento.text for elemento in raiz_bike}])

df_bicicleta.insert(0, "bike_id", "BIK_001")
df_bicicleta.insert(1, "cyclist_id", "CYC_001")

df_bicicleta

,bike_id,cyclist_id,make,model,year,size,class,isFitBike
0,BIK_001,CYC_001,Cervelo,R5,2026,56,Road,false


## 6. Lectura de los datos de la sesión


In [19]:
RUTA_SESSION = RUTA_RAW / "info.session"

raiz_session = ET.parse(RUTA_SESSION).getroot()

df_sesion = pd.DataFrame([{elemento.tag: elemento.text for elemento in raiz_session}])

df_sesion.insert(0, "session_id", "SES_001")
df_sesion.insert(1, "cyclist_id", "CYC_001")
df_sesion.insert(2, "bike_id", "BIK_001")

df_sesion

,session_id,cyclist_id,bike_id,title,startTime,isBGLevelTwo,operatorfirstname,operatorlastname,operatoremail,postSessionNotes
0,SES_001,CYC_001,BIK_001,Road,2026-06-17T17:33:50,false,Operador,Anonimo,correo@ejemplo.com,\n\t\t


## 7. Extracción de las medidas iniciales de la bicicleta

El archivo `medidas_iniciales.zin` contiene la configuración inicial de la bicicleta antes del estudio biomecánico. Al tratarse de un archivo XML con una estructura jerárquica, se utiliza una función recursiva para extraer tanto los valores simples como los elementos anidados.


In [6]:
RUTA_ZIN_INICIAL = RUTA_RAW / "medidas_iniciales.zin"

raiz_zin_inicial = ET.parse(RUTA_ZIN_INICIAL).getroot()

In [7]:
def extraer_valores_xml(elemento, prefijo=""):
    datos = {}

    for hijo in elemento:
        clave = f"{prefijo}_{hijo.tag}" if prefijo else hijo.tag

        if len(hijo) == 0:
            datos[clave] = hijo.text.strip() if hijo.text else None
        else:
            datos.update(extraer_valores_xml(hijo, clave))

    return datos


In [8]:
componentes_iniciales = extraer_valores_xml(raiz_zin_inicial.find("components"))

df_componentes_iniciales = pd.DataFrame([componentes_iniciales])

df_componentes_iniciales.insert(0, "measurement_id", "MEA_INI_001")
df_componentes_iniciales.insert(1, "session_id", "SES_001")
df_componentes_iniciales.insert(2, "phase", "Inicial")

df_componentes_iniciales


,measurement_id,session_id,phase,front_wheel,rear_wheel,saddleMake,saddleModel,crank_length,pedalMake,pedalModel,...,handlebarMake,handlebarModel,footNote,wedgeNote,cleatNote,stanceNote,armPadSpacerHeight,stem_height,stem_length,stem_angle
0,MEA_INI_001,SES_001,Inicial,699,699,Selle San Marco,Aspide,1725,Favero,Look,...,None,None,None,None,None,None,50,350,100,-8


### 7.1. Coordenadas de los puntos de la bicicleta

Las coordenadas geométricas se conservan en un DataFrame auxiliar. Permiten mantener la información técnica del archivo original, aunque no formen parte de las tablas principales del análisis en Power BI.


In [9]:
def extraer_bike_points(elemento, ruta="", registros=None):
    if registros is None:
        registros = []

    ruta_actual = f"{ruta}/{elemento.tag}" if ruta else elemento.tag

    if len(elemento) == 0:
        valor = elemento.text.strip() if elemento.text else None

        registros.append({
            "path": ruta_actual,
            "value": valor
        })
    else:
        for hijo in elemento:
            extraer_bike_points(hijo, ruta_actual, registros)

    return registros


In [10]:
registros_puntos_iniciales = []

for punto in raiz_zin_inicial.find("bikePoints"):
    registros_puntos_iniciales.extend(
        extraer_bike_points(punto)
    )

df_puntos_bicicleta_iniciales = pd.DataFrame(registros_puntos_iniciales)
df_puntos_bicicleta_iniciales.insert( 0, "measurement_id", "MEA_INI_001")
df_puntos_bicicleta_iniciales.insert(1, "session_id","SES_001")
df_puntos_bicicleta_iniciales.insert(2,"phase","Inicial")

df_puntos_bicicleta_iniciales


,measurement_id,session_id,phase,path,value
0,MEA_INI_001,SES_001,Inicial,steerer/location/xyz,-370.878 -144.557 -2300.67
1,MEA_INI_001,SES_001,Inicial,steerer/location/xyz,-370.906 -144.478 -2300.44
2,MEA_INI_001,SES_001,Inicial,steerer/location/xyz,-371.092 -144.58 -2300.78
3,MEA_INI_001,SES_001,Inicial,steerer/location/xyz,-370.985 -144.449 -2300.33
4,MEA_INI_001,SES_001,Inicial,steerer/location/xyz,-370.885 -144.338 -2299.84
...,...,...,...,...,...
331,MEA_INI_001,SES_001,Inicial,grip/contour/xyz,-538.337 -135.744 -2510.99
332,MEA_INI_001,SES_001,Inicial,grip/contour/xyz,-537.499 -135.567 -2510.43
333,MEA_INI_001,SES_001,Inicial,grip/contour/xyz,-537.046 -135.852 -2511.32
334,MEA_INI_001,SES_001,Inicial,grip/contour/xyz,-536.99 -135.608 -2510.12


## 8. Extracción de las medidas finales de la bicicleta

El archivo `medidas_finales.zin` contiene la configuración de la bicicleta después de los ajustes realizados durante el estudio biomecánico. Se reutilizan las funciones anteriores para mantener la misma estructura en los datos iniciales y finales.

In [11]:
RUTA_ZIN_FINAL = RUTA_RAW / "medidas_finales.zin"

raiz_zin_final = ET.parse(RUTA_ZIN_FINAL).getroot()

In [12]:
componentes_finales = extraer_valores_xml(raiz_zin_final.find("components"))

df_componentes_finales = pd.DataFrame([componentes_finales])

df_componentes_finales.insert(0, "measurement_id", "MEA_FIN_001")
df_componentes_finales.insert(1, "session_id", "SES_001")
df_componentes_finales.insert(2, "phase", "Final")

df_componentes_finales

,measurement_id,session_id,phase,front_wheel,rear_wheel,saddleMake,saddleModel,crank_length,pedalMake,pedalModel,...,handlebarMake,handlebarModel,footNote,wedgeNote,cleatNote,stanceNote,armPadSpacerHeight,stem_height,stem_length,stem_angle
0,MEA_FIN_001,SES_001,Final,699,699,Selle San Marco,Aspide,1725,Favero,Look,...,None,None,None,None,None,None,50,350,100,-8


### 8.1. Coordenadas finales de los puntos de la bicicleta

In [13]:
registros_puntos_finales = []

for punto in raiz_zin_final.find("bikePoints"):
    registros_puntos_finales.extend(
        extraer_bike_points(punto)
    )

df_puntos_bicicleta_finales = pd.DataFrame(registros_puntos_finales)

df_puntos_bicicleta_finales.insert(0, "measurement_id", "MEA_FIN_001")
df_puntos_bicicleta_finales.insert(1, "session_id", "SES_001")
df_puntos_bicicleta_finales.insert(2, "phase", "Final")

df_puntos_bicicleta_finales

,measurement_id,session_id,phase,path,value
0,MEA_FIN_001,SES_001,Final,steerer/location/xyz,-370.681 -144.277 -2298.62
1,MEA_FIN_001,SES_001,Final,steerer/location/xyz,-370.684 -144.202 -2298.92
2,MEA_FIN_001,SES_001,Final,steerer/location/xyz,-370.902 -144.143 -2299.27
3,MEA_FIN_001,SES_001,Final,steerer/location/xyz,-371.134 -144.145 -2299.51
4,MEA_FIN_001,SES_001,Final,steerer/location/xyz,-371.199 -144.204 -2299.25
...,...,...,...,...,...
313,MEA_FIN_001,SES_001,Final,grip/contour/xyz,-530.457 -138.063 -2507.62
314,MEA_FIN_001,SES_001,Final,grip/contour/xyz,-529.17 -138.279 -2507.97
315,MEA_FIN_001,SES_001,Final,grip/contour/xyz,-527.542 -138.277 -2507.94
316,MEA_FIN_001,SES_001,Final,grip/contour/xyz,-526.146 -137.836 -2507.31


## 9. Exportación de los datos procesados

In [20]:
dataframes_exportar = {
    "cyclist.csv": df_ciclista,
    "bike.csv": df_bicicleta,
    "session.csv": df_sesion,
    "componentes_iniciales.csv": df_componentes_iniciales,
    "componentes_finales.csv": df_componentes_finales,
    "initial_bike_points.csv": df_puntos_bicicleta_iniciales,
    "final_bike_points.csv": df_puntos_bicicleta_finales
}

for nombre_archivo, dataframe in dataframes_exportar.items():
    dataframe.to_csv(
        RUTA_PROCESSED / nombre_archivo,
        index=False
    )

## 10. Conclusiones:

Se han extraído correctamente los datos del ciclista, la bicicleta, la sesión y las medidas iniciales y finales del estudio biomecánico.

Los datos procesados se exportan en formato CSV para facilitar las siguientes fases del proyecto: análisis, almacenamiento en base de datos y visualización.

# PARTE II. GENERACIÓN Y ANÁLISIS DEL DATASET SINTÉTICO

Debido a la disponibilidad de un único estudio biomecánico, se genera un
conjunto de datos sintético formado por 100 ciclistas virtuales.

El estudio original se utiliza únicamente como plantilla estructural.
Los identificadores y valores generados no representan a personas reales.

## 11. Generación de 100 ciclistas virtuales

In [21]:
# Las tablas originales se guardan como plantillas.

plantilla_ciclista = df_ciclista.copy()
plantilla_bicicleta = df_bicicleta.copy()
plantilla_sesion = df_sesion.copy()

plantilla_componentes_iniciales = df_componentes_iniciales.copy()
plantilla_componentes_finales = df_componentes_finales.copy()

plantilla_puntos_iniciales = df_puntos_bicicleta_iniciales.copy()
plantilla_puntos_finales = df_puntos_bicicleta_finales.copy()


In [22]:
# Semilla aleatoria para reproducir el conjunto de datos.
# Permite obtener los mismos datos sintéticos en cada ejecución.

SEED = 42
NUM_CICLISTAS = 100

rng = np.random.default_rng(SEED)


In [23]:
def variar_valor(valor, variacion=0.05, minimo=None):
    try:
        numero = float(valor)
    except (TypeError, ValueError):
        return valor

    factor = rng.normal(1, variacion)
    resultado = numero * factor

    if minimo is not None:
        resultado = max(resultado, minimo)

    return round(resultado, 2)


In [24]:
nombres_hombres = [f"Ciclista_H_{i:02d}" for i in range(1, 6)]
nombres_mujeres = [f"Ciclista_M_{i:02d}" for i in range(1, 6)]

estilos_ciclismo = [
    "Recreativo",
    "Deportivo",
    "Resistencia",
    "Competitivo",
    "Élite"
]

lesiones = [
    "Ninguna",
    "Molestias de rodilla",
    "Molestias lumbares",
    "Molestias cervicales",
    "Molestias de cadera"
]

generos = np.array(
    ["Hombre"] * 85 + ["Mujer"] * 15
)
rng.shuffle(generos)

registros_ciclistas = []
columnas_ciclista = [
    columna
    for columna in plantilla_ciclista.columns
    if columna != "flexibility"
]

for numero in range(1, NUM_CICLISTAS + 1):
    id_ciclista = f"CYC_{numero:03d}"
    genero = generos[numero - 1]
    nombre = rng.choice(
        nombres_hombres if genero == "Hombre" else nombres_mujeres
    )

    grupo_edad = rng.choice(
        ["18-29", "30-39", "40-49", "50-59", "60-69"],
        p=[0.10, 0.25, 0.35, 0.22, 0.08]
    )
    limites_edad = {
        "18-29": (18, 29),
        "30-39": (30, 39),
        "40-49": (40, 49),
        "50-59": (50, 59),
        "60-69": (60, 69)
    }
    edad_minima, edad_maxima = limites_edad[grupo_edad]
    edad = int(rng.integers(edad_minima, edad_maxima + 1))
    fecha_nacimiento = datetime.today() - timedelta(
        days=edad * 365 + int(rng.integers(0, 365))
    )

    estilo = rng.choice(
        estilos_ciclismo,
        p=[0.30, 0.25, 0.20, 0.18, 0.07]
    )

    if estilo == "Recreativo":
        objetivo = rng.choice(
            ["Comodidad", "Prevención de lesiones", "Larga distancia", "Rendimiento"],
            p=[0.45, 0.30, 0.20, 0.05]
        )
        entrenamiento = rng.choice(
            ["Bajo", "Moderado", "Alto"],
            p=[0.45, 0.50, 0.05]
        )
    elif estilo == "Deportivo":
        objetivo = rng.choice(
            ["Comodidad", "Prevención de lesiones", "Larga distancia", "Rendimiento", "Competición"],
            p=[0.20, 0.25, 0.20, 0.30, 0.05]
        )
        entrenamiento = rng.choice(
            ["Bajo", "Moderado", "Alto"],
            p=[0.10, 0.65, 0.25]
        )
    elif estilo == "Resistencia":
        objetivo = rng.choice(
            ["Comodidad", "Prevención de lesiones", "Larga distancia", "Rendimiento"],
            p=[0.25, 0.20, 0.45, 0.10]
        )
        entrenamiento = rng.choice(
            ["Bajo", "Moderado", "Alto"],
            p=[0.08, 0.57, 0.35]
        )
    elif estilo == "Competitivo":
        objetivo = rng.choice(
            ["Prevención de lesiones", "Rendimiento", "Competición", "Larga distancia"],
            p=[0.15, 0.45, 0.30, 0.10]
        )
        entrenamiento = rng.choice(
            ["Moderado", "Alto"],
            p=[0.35, 0.65]
        )
    else:
        objetivo = rng.choice(
            ["Prevención de lesiones", "Rendimiento", "Competición"],
            p=[0.10, 0.55, 0.35]
        )
        entrenamiento = rng.choice(
            ["Moderado", "Alto"],
            p=[0.10, 0.90]
        )

    if objetivo in ["Comodidad", "Prevención de lesiones"]:
        lesion = rng.choice(
            lesiones,
            p=[0.42, 0.23, 0.15, 0.10, 0.10]
        )
    else:
        lesion = rng.choice(
            lesiones,
            p=[0.72, 0.12, 0.07, 0.05, 0.04]
        )

    registro = {
        columna: None
        for columna in columnas_ciclista
    }

    registro.update({
        "cyclist_id": id_ciclista,
        "lastname": f"Virtual_{numero:03d}",
        "firstname": nombre,
        "suffix": None,
        "phone": None,
        "e-mail": f"cyclist{numero:03d}@synthetic-dataset.local",
        "gender": genero,
        "birthday": fecha_nacimiento.strftime("%Y-%m-%d"),
        "ridingstyle": estilo,
        "goals": objetivo,
        "injuries": lesion,
        "training": entrenamiento
    })

    registros_ciclistas.append(registro)

df_ciclista = pd.DataFrame(
    registros_ciclistas,
    columns=columnas_ciclista
)

df_ciclista.head()

,cyclist_id,lastname,firstname,suffix,phone,e-mail,gender,birthday,ridingstyle,goals,injuries,training
0,CYC_001,Virtual_001,Ciclista_H_02,None,None,cyclist001@synthetic-dataset.local,Hombre,1992-06-01,Competitivo,Rendimiento,Ninguna,Moderado
1,CYC_002,Virtual_002,Ciclista_H_03,None,None,cyclist002@synthetic-dataset.local,Hombre,1992-01-12,Resistencia,Larga distancia,Ninguna,Alto
2,CYC_003,Virtual_003,Ciclista_H_03,None,None,cyclist003@synthetic-dataset.local,Hombre,1988-05-11,Recreativo,Prevención de lesiones,Ninguna,Moderado
3,CYC_004,Virtual_004,Ciclista_H_01,None,None,cyclist004@synthetic-dataset.local,Hombre,1977-07-20,Resistencia,Prevención de lesiones,Ninguna,Moderado
4,CYC_005,Virtual_005,Ciclista_H_04,None,None,cyclist005@synthetic-dataset.local,Hombre,1979-09-28,Recreativo,Rendimiento,Ninguna,Moderado


In [25]:
print("Dimensiones:", df_ciclista.shape)
print("Ciclistas únicos:", df_ciclista["cyclist_id"].nunique())

df_ciclista.head()


Dimensiones: (100, 12)
Ciclistas únicos: 100


,cyclist_id,lastname,firstname,suffix,phone,e-mail,gender,birthday,ridingstyle,goals,injuries,training
0,CYC_001,Virtual_001,Ciclista_H_02,None,None,cyclist001@synthetic-dataset.local,Hombre,1992-06-01,Competitivo,Rendimiento,Ninguna,Moderado
1,CYC_002,Virtual_002,Ciclista_H_03,None,None,cyclist002@synthetic-dataset.local,Hombre,1992-01-12,Resistencia,Larga distancia,Ninguna,Alto
2,CYC_003,Virtual_003,Ciclista_H_03,None,None,cyclist003@synthetic-dataset.local,Hombre,1988-05-11,Recreativo,Prevención de lesiones,Ninguna,Moderado
3,CYC_004,Virtual_004,Ciclista_H_01,None,None,cyclist004@synthetic-dataset.local,Hombre,1977-07-20,Resistencia,Prevención de lesiones,Ninguna,Moderado
4,CYC_005,Virtual_005,Ciclista_H_04,None,None,cyclist005@synthetic-dataset.local,Hombre,1979-09-28,Recreativo,Rendimiento,Ninguna,Moderado


## Diccionarios de componentes: pedales y sillines

Para representar de forma realista la configuración de la bicicleta, los pedales y los sillines se almacenan en tablas de dimensión independientes.

El archivo original contiene las variables `pedalMake`, `pedalModel`, `saddleMake` y `saddleModel`. En el modelo relacional, estos campos de texto repetidos se sustituyen por `pedal_id` y `saddle_id` en `BICICLETA`. De esta forma, cada componente permanece asociado a la bicicleta durante toda la sesión y se evita duplicar la misma información en `MEDICION`.

La tabla `PEDAL` conserva únicamente la marca y el modelo, que son las dos variables presentes en el estudio real.


In [26]:
catalogo_pedales = [
    ("PED_001", "Shimano", "PD-R550"),
    ("PED_002", "Shimano", "PD-R7000 105"),
    ("PED_003", "Shimano", "PD-R8000 Ultegra"),
    ("PED_004", "Shimano", "PD-R9100 Dura-Ace"),
    ("PED_005", "Look", "Keo 2 Max"),
    ("PED_006", "Look", "Keo Blade Carbon"),
    ("PED_007", "Favero", "Assioma Uno"),
    ("PED_008", "Favero", "Assioma Duo"),
    ("PED_009", "Garmin", "Rally RK200"),
    ("PED_010", "Garmin", "Rally RS200"),
    ("PED_011", "Wahoo", "Speedplay Nano"),
    ("PED_012", "Time", "XPRO 12")
]

df_pedal = pd.DataFrame(
    catalogo_pedales,
    columns=["pedal_id", "make", "model"]
)

probabilidades_pedales = [
    0.10, 0.16, 0.17, 0.07,
    0.10, 0.08,
    0.08, 0.07,
    0.05, 0.04,
    0.05, 0.03
]

print("Modelos de pedal disponibles:", len(df_pedal))
display(df_pedal)


Modelos de pedal disponibles: 12


,pedal_id,make,model
0,PED_001,Shimano,PD-R550
1,PED_002,Shimano,PD-R7000 105
2,PED_003,Shimano,PD-R8000 Ultegra
3,PED_004,Shimano,PD-R9100 Dura-Ace
4,PED_005,Look,Keo 2 Max
5,PED_006,Look,Keo Blade Carbon
6,PED_007,Favero,Assioma Uno
7,PED_008,Favero,Assioma Duo
8,PED_009,Garmin,Rally RK200
9,PED_010,Garmin,Rally RS200


### Diccionario de sillines

El catálogo sintético incluye marcas y modelos representativos de sillines de carretera. Cada bicicleta recibe un `saddle_id`, que permanece sin cambios durante toda la sesión biomecánica.


In [27]:
catalogo_sillines = [
    ("SAD_001", "Selle San Marco", "Aspide"),
    ("SAD_002", "Selle Italia", "SLR Boost"),
    ("SAD_003", "Fizik", "Antares R3"),
    ("SAD_004", "Prologo", "Dimension"),
    ("SAD_005", "Specialized", "Power Expert"),
    ("SAD_006", "Bontrager", "Aeolus Élite"),
    ("SAD_007", "PRO", "Stealth"),
    ("SAD_008", "SQLab", "612 Ergowave"),
    ("SAD_009", "ISM", "PN 3.1"),
    ("SAD_010", "Fabric", "Scoop Race")
]

df_sillin = pd.DataFrame(
    catalogo_sillines,
    columns=["saddle_id", "make", "model"]
)

probabilidades_sillines = [
    0.18, 0.20, 0.15, 0.14, 0.10,
    0.06, 0.06, 0.04, 0.04, 0.03
]

print("Modelos de sillín disponibles:", len(df_sillin))
display(df_sillin)


Modelos de sillín disponibles: 10


,saddle_id,make,model
0,SAD_001,Selle San Marco,Aspide
1,SAD_002,Selle Italia,SLR Boost
2,SAD_003,Fizik,Antares R3
3,SAD_004,Prologo,Dimension
4,SAD_005,Specialized,Power Expert
5,SAD_006,Bontrager,Aeolus Élite
6,SAD_007,PRO,Stealth
7,SAD_008,SQLab,612 Ergowave
8,SAD_009,ISM,PN 3.1
9,SAD_010,Fabric,Scoop Race


In [28]:
catalogo_bicicletas = [
    ("Specialized", "Tarmac SL8", "Carretera"),
    ("Specialized", "Roubaix", "Resistencia"),
    ("Trek", "Madone", "Aero"),
    ("Trek", "Emonda", "Escaladora"),
    ("Trek", "Domane", "Resistencia"),
    ("Canyon", "Ultimate CF", "Escaladora"),
    ("Canyon", "Aeroad CF", "Aero"),
    ("Canyon", "Endurace CF", "Resistencia"),
    ("Orbea", "Orca", "Carretera"),
    ("Orbea", "Orca Aero", "Aero"),
    ("Giant", "TCR Advanced", "Escaladora"),
    ("Giant", "Propel Advanced", "Aero"),
    ("Giant", "Defy Advanced", "Resistencia"),
    ("Cannondale", "SuperSix EVO", "Carretera"),
    ("Cannondale", "Synapse", "Resistencia"),
    ("BMC", "Teammachine SLR", "Carretera"),
    ("BMC", "Timemachine Carretera", "Aero"),
    ("Scott", "Addict RC", "Escaladora"),
    ("Scott", "Foil RC", "Aero"),
    ("Pinarello", "Dogma F", "Carretera"),
    ("Cervelo", "R5", "Escaladora"),
    ("Cervelo", "S5", "Aero"),
    ("Cervelo", "P-Series", "Contrarreloj"),
    ("Merida", "Scultura", "Escaladora"),
    ("Merida", "Reacto", "Aero")
]

registros_bicicletas = []

for numero in range(1, NUM_CICLISTAS + 1):
    registro = plantilla_bicicleta.iloc[0].to_dict()
    ciclista_actual = df_ciclista.iloc[numero - 1]

    if ciclista_actual["ridingstyle"] in ["Competitivo", "Élite"]:
        clases_probables = ["Carretera", "Aero", "Escaladora", "Contrarreloj"]
        probabilidades_clase = [0.30, 0.35, 0.25, 0.10]
    elif ciclista_actual["ridingstyle"] == "Resistencia":
        clases_probables = ["Resistencia", "Carretera", "Escaladora", "Aero"]
        probabilidades_clase = [0.55, 0.25, 0.12, 0.08]
    else:
        clases_probables = ["Carretera", "Resistencia", "Escaladora", "Aero"]
        probabilidades_clase = [0.40, 0.35, 0.15, 0.10]

    clase_elegida = rng.choice(
        clases_probables,
        p=probabilidades_clase
    )
    modelos_clase = [
        bicicleta
        for bicicleta in catalogo_bicicletas
        if bicicleta[2] == clase_elegida
    ]
    marca, modelo, clase_bicicleta = modelos_clase[
        int(rng.integers(0, len(modelos_clase)))
    ]

    if ciclista_actual["gender"] == "Mujer":
        talla = int(rng.choice(
            [48, 50, 52, 54, 56],
            p=[0.15, 0.25, 0.35, 0.20, 0.05]
        ))
    else:
        talla = int(rng.choice(
            [50, 52, 54, 56, 58, 60],
            p=[0.04, 0.12, 0.27, 0.32, 0.20, 0.05]
        ))

    antiguedad = int(rng.choice(
        [0, 1, 2, 3, 4, 5, 6, 7, 8],
        p=[0.12, 0.13, 0.12, 0.12, 0.12, 0.11, 0.10, 0.09, 0.09]
    ))

    registro["bike_id"] = f"BIK_{numero:03d}"
    registro["cyclist_id"] = f"CYC_{numero:03d}"
    registro["make"] = marca
    registro["model"] = modelo
    registro["year"] = 2026 - antiguedad
    registro["size"] = talla
    registro["class"] = clase_bicicleta
    registro["pedal_id"] = rng.choice(
        df_pedal["pedal_id"],
        p=probabilidades_pedales
    )
    registro["saddle_id"] = rng.choice(
        df_sillin["saddle_id"],
        p=probabilidades_sillines
    )
    registro["isFitBike"] = bool(
        rng.choice([False, True], p=[0.95, 0.05])
    )

    registros_bicicletas.append(registro)

df_bicicleta = pd.DataFrame(
    registros_bicicletas,
    columns=[*plantilla_bicicleta.columns, "pedal_id", "saddle_id"]
).reset_index(drop=True)

print("Bicicletas generadas:", len(df_bicicleta))
print("Marcas diferentes:", df_bicicleta["make"].nunique())
print("Modelos diferentes:", df_bicicleta["model"].nunique())
print("Clases diferentes:", df_bicicleta["class"].nunique())
print("Modelos de pedal utilizados:", df_bicicleta["pedal_id"].nunique())
print("Modelos de sillín utilizados:", df_bicicleta["saddle_id"].nunique())

display(df_bicicleta.head(10))

Bicicletas generadas: 100
Marcas diferentes: 11
Modelos diferentes: 24
Clases diferentes: 5
Modelos de pedal utilizados: 12
Modelos de sillín utilizados: 10


,bike_id,cyclist_id,make,model,year,size,class,isFitBike,pedal_id,saddle_id
0,BIK_001,CYC_001,Cervelo,P-Series,2023,56,Contrarreloj,False,PED_001,SAD_009
1,BIK_002,CYC_002,Specialized,Roubaix,2023,52,Resistencia,False,PED_005,SAD_002
2,BIK_003,CYC_003,Orbea,Orca,2026,54,Carretera,False,PED_007,SAD_002
3,BIK_004,CYC_004,Trek,Domane,2026,50,Resistencia,False,PED_009,SAD_001
4,BIK_005,CYC_005,Canyon,Endurace CF,2023,54,Resistencia,False,PED_008,SAD_007
5,BIK_006,CYC_006,BMC,Timemachine Carretera,2021,56,Aero,False,PED_002,SAD_002
6,BIK_007,CYC_007,Orbea,Orca,2021,56,Carretera,False,PED_002,SAD_001
7,BIK_008,CYC_008,Trek,Emonda,2026,54,Escaladora,False,PED_002,SAD_004
8,BIK_009,CYC_009,Specialized,Roubaix,2019,52,Resistencia,False,PED_001,SAD_007
9,BIK_010,CYC_010,Pinarello,Dogma F,2024,58,Carretera,False,PED_010,SAD_003


In [29]:
fecha_inicial = datetime(2024, 1, 1)
fecha_final = datetime(2026, 7, 30)
franjas_horarias = [
    (9, 0), (9, 30), (10, 0), (10, 30),
    (11, 0), (11, 30), (12, 0), (12, 30),
    (15, 0), (15, 30), (16, 0), (16, 30),
    (17, 0), (17, 30)
]

registros_sesiones = []
columnas_sesion = [
    columna
    for columna in plantilla_sesion.columns
    if columna != "isBGLevelTwo"
]

for numero in range(1, NUM_CICLISTAS + 1):
    registro = {
        columna: plantilla_sesion.iloc[0][columna]
        for columna in columnas_sesion
    }
    objetivo = df_ciclista.loc[numero - 1, "goals"]

    titulos_por_objetivo = {
        "Comodidad": ["Comodidad", "Resistencia"],
        "Prevención de lesiones": ["Comodidad", "Resistencia"],
        "Larga distancia": ["Resistencia", "Carretera"],
        "Rendimiento": ["Rendimiento", "Carretera"],
        "Competición": ["Competición", "Rendimiento"]
    }

    dias_disponibles = (fecha_final - fecha_inicial).days
    fecha_sesion = fecha_inicial + timedelta(
        days=int(rng.integers(0, dias_disponibles + 1))
    )

    while fecha_sesion.weekday() >= 5:
        fecha_sesion += timedelta(days=1)
        if fecha_sesion > fecha_final:
            fecha_sesion -= timedelta(days=3)

    hora, minuto = franjas_horarias[
        int(rng.integers(0, len(franjas_horarias)))
    ]
    fecha_sesion = fecha_sesion.replace(
        hour=hora,
        minute=minuto,
        second=0
    )

    registro["session_id"] = f"SES_{numero:03d}"
    registro["cyclist_id"] = f"CYC_{numero:03d}"
    registro["bike_id"] = f"BIK_{numero:03d}"
    registro["title"] = rng.choice(titulos_por_objetivo[objetivo])
    registro["startTime"] = fecha_sesion.strftime("%Y-%m-%dT%H:%M:%S")

    registros_sesiones.append(registro)

df_sesion = pd.DataFrame(
    registros_sesiones,
    columns=columnas_sesion
).reset_index(drop=True)

df_sesion.head()

,session_id,cyclist_id,bike_id,title,startTime,operatorfirstname,operatorlastname,operatoremail,postSessionNotes
0,SES_001,CYC_001,BIK_001,Carretera,2025-02-25T09:30:00,Operador,Anonimo,correo@ejemplo.com,\n\t\t
1,SES_002,CYC_002,BIK_002,Resistencia,2024-06-26T12:00:00,Operador,Anonimo,correo@ejemplo.com,\n\t\t
2,SES_003,CYC_003,BIK_003,Resistencia,2024-06-24T09:30:00,Operador,Anonimo,correo@ejemplo.com,\n\t\t
3,SES_004,CYC_004,BIK_004,Comodidad,2026-06-11T10:00:00,Operador,Anonimo,correo@ejemplo.com,\n\t\t
4,SES_005,CYC_005,BIK_005,Rendimiento,2024-07-30T16:30:00,Operador,Anonimo,correo@ejemplo.com,\n\t\t


In [30]:
columnas_identificadoras = {
    "measurement_id",
    "session_id",
    "phase"
}

registros_componentes_iniciales = []

for numero in range(1, NUM_CICLISTAS + 1):
    registro = plantilla_componentes_iniciales.iloc[0].copy()
    talla_bicicleta = int(df_bicicleta.loc[numero - 1, "size"])

    registro["measurement_id"] = f"MEA_INI_{numero:03d}"
    registro["session_id"] = f"SES_{numero:03d}"
    registro["phase"] = "Inicial"

    registro["front_wheel"] = round(float(rng.normal(699, 3)), 2)
    registro["rear_wheel"] = round(float(rng.normal(699, 3)), 2)

    if talla_bicicleta <= 50:
        longitudes_biela = [160.0, 165.0, 167.5]
        probabilidades_biela = [0.20, 0.55, 0.25]
        longitudes_potencia = [70.0, 80.0, 90.0]
        probabilidades_potencia = [0.20, 0.50, 0.30]
    elif talla_bicicleta <= 54:
        longitudes_biela = [165.0, 167.5, 170.0, 172.5]
        probabilidades_biela = [0.10, 0.25, 0.45, 0.20]
        longitudes_potencia = [80.0, 90.0, 100.0, 110.0]
        probabilidades_potencia = [0.10, 0.35, 0.40, 0.15]
    else:
        longitudes_biela = [170.0, 172.5, 175.0, 177.5]
        probabilidades_biela = [0.15, 0.40, 0.35, 0.10]
        longitudes_potencia = [90.0, 100.0, 110.0, 120.0]
        probabilidades_potencia = [0.15, 0.35, 0.35, 0.15]

    registro["crank_length"] = float(rng.choice(
        longitudes_biela,
        p=probabilidades_biela
    ))
    registro["armPadSpacerHeight"] = round(
        float(np.clip(rng.normal(50, 5), 35, 65)),
        2
    )
    registro["stem_height"] = round(
        float(np.clip(rng.normal(350, 22), 290, 410)),
        2
    )
    registro["stem_length"] = float(rng.choice(
        longitudes_potencia,
        p=probabilidades_potencia
    ))
    registro["stem_angle"] = float(rng.choice(
        [-12.0, -10.0, -8.0, -6.0],
        p=[0.08, 0.17, 0.35, 0.40]
    ))

    registros_componentes_iniciales.append(registro)

df_componentes_iniciales = pd.DataFrame(
    registros_componentes_iniciales
).reset_index(drop=True)

df_componentes_iniciales.head()

,measurement_id,session_id,phase,front_wheel,rear_wheel,saddleMake,saddleModel,crank_length,pedalMake,pedalModel,...,handlebarMake,handlebarModel,footNote,wedgeNote,cleatNote,stanceNote,armPadSpacerHeight,stem_height,stem_length,stem_angle
0,MEA_INI_001,SES_001,Inicial,694.63,697.83,Selle San Marco,Aspide,172.5,Favero,Look,...,None,None,None,None,None,None,52.25,362.39,120.0,-6.0
1,MEA_INI_002,SES_002,Inicial,701.33,696.57,Selle San Marco,Aspide,170.0,Favero,Look,...,None,None,None,None,None,None,43.13,357.22,90.0,-8.0
2,MEA_INI_003,SES_003,Inicial,698.93,697.13,Selle San Marco,Aspide,170.0,Favero,Look,...,None,None,None,None,None,None,48.23,325.71,100.0,-6.0
3,MEA_INI_004,SES_004,Inicial,701.14,695.46,Selle San Marco,Aspide,167.5,Favero,Look,...,None,None,None,None,None,None,46.88,379.15,70.0,-8.0
4,MEA_INI_005,SES_005,Inicial,700.50,692.68,Selle San Marco,Aspide,170.0,Favero,Look,...,None,None,None,None,None,None,59.99,352.91,90.0,-8.0


In [31]:
registros_componentes_finales = []

for numero in range(1, NUM_CICLISTAS + 1):
    registro_inicial = df_componentes_iniciales.iloc[numero - 1].copy()
    registro_final = registro_inicial.copy()

    registro_final["measurement_id"] = f"MEA_FIN_{numero:03d}"
    registro_final["session_id"] = f"SES_{numero:03d}"
    registro_final["phase"] = "Final"

    if rng.random() < 0.80:
        registro_final["stem_height"] = round(
            float(registro_inicial["stem_height"]) +
            float(rng.choice([-15, -10, -5, 5, 10, 15])),
            2
        )

    if rng.random() < 0.55:
        registro_final["stem_length"] = float(np.clip(
            float(registro_inicial["stem_length"]) +
            float(rng.choice([-20, -10, 10, 20])),
            70,
            130
        ))

    if rng.random() < 0.35:
        angulos_disponibles = [-12.0, -10.0, -8.0, -6.0]
        angulos_alternativos = [
            angulo
            for angulo in angulos_disponibles
            if angulo != float(registro_inicial["stem_angle"])
        ]
        registro_final["stem_angle"] = float(
            rng.choice(angulos_alternativos)
        )

    if rng.random() < 0.45:
        registro_final["armPadSpacerHeight"] = round(
            float(np.clip(
                float(registro_inicial["armPadSpacerHeight"]) +
                float(rng.choice([-10, -5, 5, 10])),
                25,
                75
            )),
            2
        )

    if rng.random() < 0.12:
        longitudes_biela = [160.0, 165.0, 167.5, 170.0, 172.5, 175.0, 177.5]
        posicion_actual = longitudes_biela.index(
            float(registro_inicial["crank_length"])
        )
        posiciones_posibles = [
            posicion
            for posicion in [posicion_actual - 1, posicion_actual + 1]
            if 0 <= posicion < len(longitudes_biela)
        ]
        registro_final["crank_length"] = longitudes_biela[
            int(rng.choice(posiciones_posibles))
        ]

    registros_componentes_finales.append(registro_final)

df_componentes_finales = pd.DataFrame(
    registros_componentes_finales
).reset_index(drop=True)

df_componentes_finales.head()

,measurement_id,session_id,phase,front_wheel,rear_wheel,saddleMake,saddleModel,crank_length,pedalMake,pedalModel,...,handlebarMake,handlebarModel,footNote,wedgeNote,cleatNote,stanceNote,armPadSpacerHeight,stem_height,stem_length,stem_angle
0,MEA_FIN_001,SES_001,Final,694.63,697.83,Selle San Marco,Aspide,172.5,Favero,Look,...,None,None,None,None,None,None,57.25,367.39,130.0,-6.0
1,MEA_FIN_002,SES_002,Final,701.33,696.57,Selle San Marco,Aspide,170.0,Favero,Look,...,None,None,None,None,None,None,43.13,342.22,90.0,-6.0
2,MEA_FIN_003,SES_003,Final,698.93,697.13,Selle San Marco,Aspide,167.5,Favero,Look,...,None,None,None,None,None,None,43.23,340.71,120.0,-6.0
3,MEA_FIN_004,SES_004,Final,701.14,695.46,Selle San Marco,Aspide,167.5,Favero,Look,...,None,None,None,None,None,None,46.88,374.15,70.0,-12.0
4,MEA_FIN_005,SES_005,Final,700.50,692.68,Selle San Marco,Aspide,170.0,Favero,Look,...,None,None,None,None,None,None,59.99,362.91,90.0,-8.0


Creación de los puntos iniciales

In [32]:
lista_puntos_iniciales = []

for numero in range(1, NUM_CICLISTAS + 1):

    puntos_ciclista = (
        plantilla_puntos_iniciales.copy()
    )

    puntos_ciclista["measurement_id"] = f"MEA_INI_{numero:03d}"
    puntos_ciclista["session_id"] = f"SES_{numero:03d}"
    puntos_ciclista["phase"] = "Inicial"

    puntos_ciclista["value"] = puntos_ciclista["value"].apply(
        lambda valor: variar_valor(valor, variacion=0.06)
    )

    lista_puntos_iniciales.append(puntos_ciclista)

df_puntos_bicicleta_iniciales = pd.concat(
    lista_puntos_iniciales,
    ignore_index=True
)

df_puntos_bicicleta_iniciales.head()

,measurement_id,session_id,phase,path,value
0,MEA_INI_001,SES_001,Inicial,steerer/location/xyz,-370.878 -144.557 -2300.67
1,MEA_INI_001,SES_001,Inicial,steerer/location/xyz,-370.906 -144.478 -2300.44
2,MEA_INI_001,SES_001,Inicial,steerer/location/xyz,-371.092 -144.58 -2300.78
3,MEA_INI_001,SES_001,Inicial,steerer/location/xyz,-370.985 -144.449 -2300.33
4,MEA_INI_001,SES_001,Inicial,steerer/location/xyz,-370.885 -144.338 -2299.84


Creación de los puntos finales

In [33]:
# Se utiliza un generador independiente para no alterar el resto de la semilla del dataset.
rng_sillin = np.random.default_rng(4201)

# Probabilidad de que el estudio biomecánico requiera modificar la posición del sillín.
# Es una hipótesis de simulación del dataset sintético, no un resultado del caso real.
probabilidad_cambio_sillin = 0.70

cambio_sillin_por_sesion = {
    f"SES_{numero:03d}": bool(rng_sillin.random() < probabilidad_cambio_sillin)
    for numero in range(1, NUM_CICLISTAS + 1)
}

lista_puntos_finales = []

for numero in range(1, NUM_CICLISTAS + 1):

    sesion_id = f"SES_{numero:03d}"

    puntos_ciclista = plantilla_puntos_finales.copy()

    puntos_ciclista["measurement_id"] = f"MEA_FIN_{numero:03d}"
    puntos_ciclista["session_id"] = sesion_id
    puntos_ciclista["phase"] = "Final"

    puntos_ciclista["value"] = puntos_ciclista["value"].apply(
        lambda valor: variar_valor(valor, variacion=0.02)
    )

    # Si el sillín no cambia, se conserva su posición inicial alineando
    # el centro de cada coordenada del seat_point y del contorno del sillín.
    if not cambio_sillin_por_sesion[sesion_id]:
        puntos_iniciales_sesion = df_puntos_bicicleta_iniciales[
            df_puntos_bicicleta_iniciales["session_id"] == sesion_id
        ]

        mascara_sillin_final = puntos_ciclista["path"].str.contains(
            "seat_point|saddle",
            case=False,
            regex=True
        )

        for ruta in puntos_ciclista.loc[mascara_sillin_final, "path"].unique():
            valores_iniciales = pd.to_numeric(
                puntos_iniciales_sesion.loc[
                    puntos_iniciales_sesion["path"] == ruta,
                    "value"
                ],
                errors="coerce"
            )
            valores_finales = pd.to_numeric(
                puntos_ciclista.loc[puntos_ciclista["path"] == ruta, "value"],
                errors="coerce"
            )

            if valores_iniciales.notna().any() and valores_finales.notna().any():
                desplazamiento = valores_iniciales.mean() - valores_finales.mean()
                puntos_ciclista.loc[
                    puntos_ciclista["path"] == ruta,
                    "value"
                ] = (valores_finales + desplazamiento).round(2).values

    lista_puntos_finales.append(puntos_ciclista)

df_puntos_bicicleta_finales = pd.concat(
    lista_puntos_finales,
    ignore_index=True
)

df_puntos_bicicleta_finales.head()


,measurement_id,session_id,phase,path,value
0,MEA_FIN_001,SES_001,Final,steerer/location/xyz,-370.681 -144.277 -2298.62
1,MEA_FIN_001,SES_001,Final,steerer/location/xyz,-370.684 -144.202 -2298.92
2,MEA_FIN_001,SES_001,Final,steerer/location/xyz,-370.902 -144.143 -2299.27
3,MEA_FIN_001,SES_001,Final,steerer/location/xyz,-371.134 -144.145 -2299.51
4,MEA_FIN_001,SES_001,Final,steerer/location/xyz,-371.199 -144.204 -2299.25


Validación general

### Validación del cambio de posición del sillín

Se comprueba cuántas sesiones presentan una modificación de la posición del sillín entre la medición inicial y final. La probabilidad de cambio se ha fijado en 0,70 como hipótesis de simulación y se utiliza un generador aleatorio independiente para mantener la reproducibilidad sin alterar el resto del dataset.


In [34]:
resumen_cambio_sillin = pd.DataFrame({
    "session_id": list(cambio_sillin_por_sesion.keys()),
    "cambio_sillin": list(cambio_sillin_por_sesion.values())
})

numero_cambios_sillin = int(resumen_cambio_sillin["cambio_sillin"].sum())
porcentaje_cambio_sillin = round(
    resumen_cambio_sillin["cambio_sillin"].mean() * 100,
    1
)

print("Sesiones con cambio de posición del sillín:", numero_cambios_sillin)
print("Porcentaje de cambio de posición del sillín:", f"{porcentaje_cambio_sillin}%")

resumen_cambio_sillin.head()


Sesiones con cambio de posición del sillín: 72
Porcentaje de cambio de posición del sillín: 72.0%


,session_id,cambio_sillin
0,SES_001,True
1,SES_002,False
2,SES_003,False
3,SES_004,True
4,SES_005,True


In [35]:
dataframes_sinteticos = {
    "Ciclistas": df_ciclista,
    "Bicicletas": df_bicicleta,
    "Pedales": df_pedal,
    "Sillines": df_sillin,
    "Sesiones": df_sesion,
    "Componentes iniciales": df_componentes_iniciales,
    "Componentes finales": df_componentes_finales,
    "Puntos iniciales de la bicicleta": df_puntos_bicicleta_iniciales,
    "Puntos finales de la bicicleta": df_puntos_bicicleta_finales
}

for nombre_df, dataframe in dataframes_sinteticos.items():
    print(f"{nombre_df}: {dataframe.shape}")


Ciclistas: (100, 12)
Bicicletas: (100, 10)
Pedales: (12, 3)
Sillines: (10, 3)
Sesiones: (100, 9)
Componentes iniciales: (100, 22)
Componentes finales: (100, 22)
Puntos iniciales de la bicicleta: (33600, 5)
Puntos finales de la bicicleta: (31800, 5)


### Resultado de la validación del dataset sintético

Se ha comprobado la integridad referencial y la unicidad de los identificadores
generados para el conjunto de datos sintético.

Las validaciones realizadas muestran los siguientes resultados:

- Todos los ciclistas tienen una bicicleta asociada.
- Todas las bicicletas tienen un pedal válido asociado mediante `pedal_id`.
- Todos los ciclistas tienen una sesión asociada.
- Todas las sesiones disponen de una medición inicial.
- Todas las sesiones disponen de una medición final.
- Los identificadores de ciclistas (cyclist_id) son únicos.
- Los identificadores de bicicletas (bike_id) son únicos.
- Los identificadores de sesiones (session_id) son únicos.
- Los identificadores de las mediciones iniciales (measurement_id) son únicos.
- Los identificadores de las mediciones finales (measurement_id) son únicos.

Además, no se han detectado registros duplicados ni registros huérfanos,
garantizando la consistencia del modelo de datos antes de continuar con el
análisis y la exportación de los datos.

In [36]:
print("Ciclistas únicos:", df_ciclista["cyclist_id"].nunique())
print("Bicicletas únicas:", df_bicicleta["bike_id"].nunique())
print("Sesiones únicas:", df_sesion["session_id"].nunique())
print("Mediciones iniciales únicas:", df_componentes_iniciales["measurement_id"].nunique())
print("Mediciones finales únicas:", df_componentes_finales["measurement_id"].nunique())

print()

print("Ciclistas sin bike:", len(set(df_ciclista["cyclist_id"]) - set(df_bicicleta["cyclist_id"])))
print("Ciclistas sin sesión:", len(set(df_ciclista["cyclist_id"]) - set(df_sesion["cyclist_id"])))
print("Sesiones sin medición inicial:", len(set(df_sesion["session_id"]) - set(df_componentes_iniciales["session_id"])))
print("Sesiones sin medición final:", len(set(df_sesion["session_id"]) - set(df_componentes_finales["session_id"])))
print("Bicicletas sin df_pedal válido:", len(
    set(df_bicicleta["pedal_id"]) - set(df_pedal["pedal_id"])
))


Ciclistas únicos: 100
Bicicletas únicas: 100
Sesiones únicas: 100
Mediciones iniciales únicas: 100
Mediciones finales únicas: 100

Ciclistas sin bike: 0
Ciclistas sin sesión: 0
Sesiones sin medición inicial: 0
Sesiones sin medición final: 0
Bicicletas sin df_pedal válido: 0


### Validación de la variedad de bicicletas

Se comprueba que los 100 perfiles no utilizan una única bicicleta. Las marcas, los modelos, los años, las tallas y las clases se han generado de forma sintética y reproducible.


In [37]:
print("Clases diferentes:", df_bicicleta["class"].nunique())
print("Duplicados completos:", df_bicicleta.duplicated().sum())

display(
    df_bicicleta.groupby(
        ["make", "model", "class"]
    )
    .size()
    .reset_index(name="bike_count")
    .sort_values(
        ["bike_count", "make", "model"],
        ascending=[False, True, True]
    )
)

Clases diferentes: 5
Duplicados completos: 0


,make,model,class,bike_count
5,Canyon,Endurace CF,Resistencia,8
18,Scott,Addict RC,Escaladora,8
15,Orbea,Orca,Carretera,7
21,Specialized,Tarmac SL8,Carretera,7
17,Pinarello,Dogma F,Carretera,6
22,Trek,Domane,Resistencia,6
2,Cannondale,SuperSix EVO,Carretera,5
3,Cannondale,Synapse,Resistencia,5
10,Giant,Defy Advanced,Resistencia,5
16,Orbea,Orca Aero,Aero,5


## Fin de la creación del dataset sintético

Con esta etapa finaliza la generación del conjunto de datos sintético a partir del estudio biomecánico original.

Se han creado y validado las tablas necesarias para el proyecto, garantizando la integridad referencial y la unicidad de los identificadores en todas las entidades principales.

El conjunto de datos resultante está formado por 100 ciclistas virtuales y sus correspondientes bicicletas, sesiones y mediciones biomecánicas, manteniendo una estructura relacional consistente y preparada para su explotación.

A partir de este punto, el proyecto se centrará en el análisis exploratorio de los datos, la obtención de indicadores y la construcción del modelo analítico en SQL y Power BI.

# Parte III: Modelado de datos

- 1 ciclista → 1 bicicleta → 1 sesión biomecánica.
- Cada bicicleta está asociada a un modelo del diccionario `PEDAL`.
- Cada sesión contiene 1 medición inicial, 1 medición final y numerosos puntos tridimensionales.
- El pedal se registra como componente de la bicicleta, por lo que no se duplica en las mediciones inicial y final.


## Limpieza de datos: eliminaremos las columnas del archivo real que no sean necesarias para el estudio

df_cyclist: eliminamos 3 columnas, convertimos "birthdate" a fecha, lo guardamos en ciclista para conservar el origen de los datos

### Transformación de la tabla `ciclista`

Se eliminan los campos personales no necesarios para el análisis. La variable `flexibility` también se elimina porque no es relevante para los objetivos del proyecto.

In [38]:
ciclista = df_ciclista.drop(
    columns=["suffix", "phone", "e-mail"],
    errors="ignore"
)
ciclista["birthday"] = pd.to_datetime(ciclista["birthday"])
ciclista.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   cyclist_id   100 non-null    str           
 1   lastname     100 non-null    str           
 2   firstname    100 non-null    str           
 3   gender       100 non-null    str           
 4   birthday     100 non-null    datetime64[us]
 5   ridingstyle  100 non-null    str           
 6   goals        100 non-null    str           
 7   injuries     100 non-null    str           
 8   training     100 non-null    str           
dtypes: datetime64[us](1), str(8)
memory usage: 7.2 KB


### Preparación de `df_bike`

Se mantienen las variables descriptivas de la bicicleta, se convierten los tipos de datos necesarios y se conserva `pedal_id` como clave foránea hacia el diccionario `PEDAL`.


In [39]:
df_bicicleta.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   bike_id     100 non-null    str  
 1   cyclist_id  100 non-null    str  
 2   make        100 non-null    str  
 3   model       100 non-null    str  
 4   year        100 non-null    int64
 5   size        100 non-null    int64
 6   class       100 non-null    str  
 7   isFitBike   100 non-null    bool 
 8   pedal_id    100 non-null    str  
 9   saddle_id   100 non-null    str  
dtypes: bool(1), int64(2), str(7)
memory usage: 7.3 KB


In [40]:
bicicleta = df_bicicleta.copy()

bicicleta["year"] = pd.to_numeric(
    bicicleta["year"],
    errors="raise"
).astype(int)

bicicleta["size"] = pd.to_numeric(
    bicicleta["size"],
    errors="raise"
).astype(int)

bicicleta["isFitBike"] = (
    bicicleta["isFitBike"]
    .replace({
        "true": True,
        "false": False,
        "True": True,
        "False": False
    })
    .astype(bool)
)

bicicleta.info()
display(bicicleta.head())


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   bike_id     100 non-null    str  
 1   cyclist_id  100 non-null    str  
 2   make        100 non-null    str  
 3   model       100 non-null    str  
 4   year        100 non-null    int64
 5   size        100 non-null    int64
 6   class       100 non-null    str  
 7   isFitBike   100 non-null    bool 
 8   pedal_id    100 non-null    str  
 9   saddle_id   100 non-null    str  
dtypes: bool(1), int64(2), str(7)
memory usage: 7.3 KB


,bike_id,cyclist_id,make,model,year,size,class,isFitBike,pedal_id,saddle_id
0,BIK_001,CYC_001,Cervelo,P-Series,2023,56,Contrarreloj,False,PED_001,SAD_009
1,BIK_002,CYC_002,Specialized,Roubaix,2023,52,Resistencia,False,PED_005,SAD_002
2,BIK_003,CYC_003,Orbea,Orca,2026,54,Carretera,False,PED_007,SAD_002
3,BIK_004,CYC_004,Trek,Domane,2026,50,Resistencia,False,PED_009,SAD_001
4,BIK_005,CYC_005,Canyon,Endurace CF,2023,54,Resistencia,False,PED_008,SAD_007


df_session:  
- `startTime` representa una fecha y hora, por lo que se convierte de texto a `datetime`.  
- `operatorfirstname`, `operatorlastname`, `operatoremail` y `postSessionNotes` se eliminan porque no aportan información analítica.  
- `isBGLevelTwo` se elimina del modelo porque no es relevante para los objetivos del proyecto.

In [41]:
df_sesion.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   session_id         100 non-null    str  
 1   cyclist_id         100 non-null    str  
 2   bike_id            100 non-null    str  
 3   title              100 non-null    str  
 4   startTime          100 non-null    str  
 5   operatorfirstname  100 non-null    str  
 6   operatorlastname   100 non-null    str  
 7   operatoremail      100 non-null    str  
 8   postSessionNotes   100 non-null    str  
dtypes: str(9)
memory usage: 7.2 KB


In [42]:
sesion = df_sesion.copy()

sesion["startTime"] = pd.to_datetime(sesion["startTime"])

sesion = sesion.drop(
    columns=[
        "isBGLevelTwo",
        "operatorfirstname",
        "operatorlastname",
        "operatoremail",
        "postSessionNotes"
    ],
    errors="ignore"
)

sesion.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   session_id  100 non-null    str           
 1   cyclist_id  100 non-null    str           
 2   bike_id     100 non-null    str           
 3   title       100 non-null    str           
 4   startTime   100 non-null    datetime64[us]
dtypes: datetime64[us](1), str(4)
memory usage: 4.0 KB


df_initial_components:
 - convertimos las medidas numéricas a tipo float,
 - eliminamos las columnas sin información o completamente nulas,
 

In [43]:
columnas_float = [
    "front_wheel",
    "rear_wheel",
    "crank_length",
    "armPadSpacerHeight",
    "stem_height",
    "stem_length",
    "stem_angle"
]

df_componentes_iniciales[columnas_float] = (
    df_componentes_iniciales[columnas_float].astype(float)
)

columnas_vacias = [
    "shoeMake",
    "shoeModel",
    "handlebarMake",
    "handlebarModel",
    "footNote",
    "wedgeNote",
    "cleatNote",
    "stanceNote"
]

df_componentes_iniciales.drop(
    columns=columnas_vacias,
    inplace=True,
    errors="ignore"
)

df_componentes_iniciales.info()

print("\nValores nulos:")
display(df_componentes_iniciales.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   measurement_id      100 non-null    str    
 1   session_id          100 non-null    str    
 2   phase               100 non-null    str    
 3   front_wheel         100 non-null    float64
 4   rear_wheel          100 non-null    float64
 5   saddleMake          100 non-null    str    
 6   saddleModel         100 non-null    str    
 7   crank_length        100 non-null    float64
 8   pedalMake           100 non-null    str    
 9   pedalModel          100 non-null    str    
 10  armPadSpacerHeight  100 non-null    float64
 11  stem_height         100 non-null    float64
 12  stem_length         100 non-null    float64
 13  stem_angle          100 non-null    float64
dtypes: float64(7), str(7)
memory usage: 11.1 KB

Valores nulos:


measurement_id        0
session_id            0
phase                 0
front_wheel           0
rear_wheel            0
saddleMake            0
saddleModel           0
crank_length          0
pedalMake             0
pedalModel            0
armPadSpacerHeight    0
stem_height           0
stem_length           0
stem_angle            0
dtype: int64

### Limpieza de los componentes finales

En `df_componentes_finales` se aplica el mismo proceso de transformación que en los componentes iniciales:

- se convierten las medidas numéricas a tipo `float`;
- se eliminan las columnas completamente vacías;
- se comprueban los tipos de datos y los valores nulos resultantes.


In [44]:
columnas_float = [
    "front_wheel",
    "rear_wheel",
    "crank_length",
    "armPadSpacerHeight",
    "stem_height",
    "stem_length",
    "stem_angle"
]

df_componentes_finales[columnas_float] = (
    df_componentes_finales[columnas_float].astype(float)
)

columnas_vacias = [
    "shoeMake",
    "shoeModel",
    "handlebarMake",
    "handlebarModel",
    "footNote",
    "wedgeNote",
    "cleatNote",
    "stanceNote"
]

df_componentes_finales.drop(
    columns=columnas_vacias,
    inplace=True,
    errors="ignore"
)

df_componentes_finales.info()

print("\nValores nulos:")
display(df_componentes_finales.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   measurement_id      100 non-null    str    
 1   session_id          100 non-null    str    
 2   phase               100 non-null    str    
 3   front_wheel         100 non-null    float64
 4   rear_wheel          100 non-null    float64
 5   saddleMake          100 non-null    str    
 6   saddleModel         100 non-null    str    
 7   crank_length        100 non-null    float64
 8   pedalMake           100 non-null    str    
 9   pedalModel          100 non-null    str    
 10  armPadSpacerHeight  100 non-null    float64
 11  stem_height         100 non-null    float64
 12  stem_length         100 non-null    float64
 13  stem_angle          100 non-null    float64
dtypes: float64(7), str(7)
memory usage: 11.1 KB

Valores nulos:


measurement_id        0
session_id            0
phase                 0
front_wheel           0
rear_wheel            0
saddleMake            0
saddleModel           0
crank_length          0
pedalMake             0
pedalModel            0
armPadSpacerHeight    0
stem_height           0
stem_length           0
stem_angle            0
dtype: int64

El software de captura contempla variables adicionales relacionadas con el calzado, el manillar y determinadas observaciones biomecánicas. Las columnas completamente vacías se eliminan durante el proceso ETL.

Los campos originales `pedalMake` y `pedalModel` también se retiran de `MEDICION`: su contenido se ha reinterpretado y normalizado en la tabla `PEDAL`. El modelo concreto se relaciona con cada bicicleta mediante `pedal_id`, evitando repetir el mismo componente en las mediciones inicial y final.


In [45]:
medicion_inicial = df_componentes_iniciales.copy()

columnas_float = [
    "front_wheel",
    "rear_wheel",
    "crank_length",
    "armPadSpacerHeight",
    "stem_height",
    "stem_length",
    "stem_angle"
]

medicion_inicial[columnas_float] = medicion_inicial[columnas_float].astype(float)

medicion_inicial = medicion_inicial.drop(
    columns=[
        "phase",
        "shoeMake",
        "shoeModel",
        "handlebarMake",
        "handlebarModel",
        "footNote",
        "wedgeNote",
        "cleatNote",
        "stanceNote",
        "pedalMake",
        "pedalModel",
        "saddleMake",
        "saddleModel"
    ],
    errors="ignore"
)

medicion_inicial.info()
display(medicion_inicial.head())


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   measurement_id      100 non-null    str    
 1   session_id          100 non-null    str    
 2   front_wheel         100 non-null    float64
 3   rear_wheel          100 non-null    float64
 4   crank_length        100 non-null    float64
 5   armPadSpacerHeight  100 non-null    float64
 6   stem_height         100 non-null    float64
 7   stem_length         100 non-null    float64
 8   stem_angle          100 non-null    float64
dtypes: float64(7), str(2)
memory usage: 7.2 KB


,measurement_id,session_id,front_wheel,rear_wheel,crank_length,armPadSpacerHeight,stem_height,stem_length,stem_angle
0,MEA_INI_001,SES_001,694.63,697.83,172.5,52.25,362.39,120.0,-6.0
1,MEA_INI_002,SES_002,701.33,696.57,170.0,43.13,357.22,90.0,-8.0
2,MEA_INI_003,SES_003,698.93,697.13,170.0,48.23,325.71,100.0,-6.0
3,MEA_INI_004,SES_004,701.14,695.46,167.5,46.88,379.15,70.0,-8.0
4,MEA_INI_005,SES_005,700.50,692.68,170.0,59.99,352.91,90.0,-8.0


In [46]:
medicion_final = df_componentes_finales.copy()

columnas_float = [
    "front_wheel",
    "rear_wheel",
    "crank_length",
    "armPadSpacerHeight",
    "stem_height",
    "stem_length",
    "stem_angle"
]

medicion_final[columnas_float] = medicion_final[columnas_float].astype(float)

medicion_final = medicion_final.drop(
    columns=[
        "phase",
        "shoeMake",
        "shoeModel",
        "handlebarMake",
        "handlebarModel",
        "footNote",
        "wedgeNote",
        "cleatNote",
        "stanceNote",
        "pedalMake",
        "pedalModel",
        "saddleMake",
        "saddleModel"
    ],
    errors="ignore"
)

medicion_final.info()
display(medicion_final.head())


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   measurement_id      100 non-null    str    
 1   session_id          100 non-null    str    
 2   front_wheel         100 non-null    float64
 3   rear_wheel          100 non-null    float64
 4   crank_length        100 non-null    float64
 5   armPadSpacerHeight  100 non-null    float64
 6   stem_height         100 non-null    float64
 7   stem_length         100 non-null    float64
 8   stem_angle          100 non-null    float64
dtypes: float64(7), str(2)
memory usage: 7.2 KB


,measurement_id,session_id,front_wheel,rear_wheel,crank_length,armPadSpacerHeight,stem_height,stem_length,stem_angle
0,MEA_FIN_001,SES_001,694.63,697.83,172.5,57.25,367.39,130.0,-6.0
1,MEA_FIN_002,SES_002,701.33,696.57,170.0,43.13,342.22,90.0,-6.0
2,MEA_FIN_003,SES_003,698.93,697.13,167.5,43.23,340.71,120.0,-6.0
3,MEA_FIN_004,SES_004,701.14,695.46,167.5,46.88,374.15,70.0,-12.0
4,MEA_FIN_005,SES_005,700.50,692.68,170.0,59.99,362.91,90.0,-8.0


### Transformación de los puntos iniciales de la bicicleta

En esta transformación se prepara la tabla que almacena las coordenadas tridimensionales de la bicicleta durante la medición inicial:

- se añade `point_id` como identificador único de cada registro;
- se elimina `phase`, ya que contiene un único valor constante (`Inicial`);
- el contenido de `value` se divide en tres columnas numéricas: `x`, `y` y `z`.


In [47]:
df_puntos_bicicleta_iniciales.info()

<class 'pandas.DataFrame'>
RangeIndex: 33600 entries, 0 to 33599
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   measurement_id  33600 non-null  str  
 1   session_id      33600 non-null  str  
 2   phase           33600 non-null  str  
 3   path            33600 non-null  str  
 4   value           33600 non-null  str  
dtypes: str(5)
memory usage: 1.3 MB


In [48]:
puntos_bicicleta_iniciales = df_puntos_bicicleta_iniciales.copy()
puntos_bicicleta_iniciales.insert(
    0,
    "point_id",
    [f"PNT_INI_{i:06d}" for i in range(1, len(puntos_bicicleta_iniciales) + 1)]
)

puntos_bicicleta_iniciales[["x", "y", "z"]] = (
    puntos_bicicleta_iniciales["value"]
    .str.split(expand=True)
    .astype(float)
)

puntos_bicicleta_iniciales = puntos_bicicleta_iniciales.drop(columns=["phase","value"])

puntos_bicicleta_iniciales.info()

<class 'pandas.DataFrame'>
RangeIndex: 33600 entries, 0 to 33599
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   point_id        33600 non-null  str    
 1   measurement_id  33600 non-null  str    
 2   session_id      33600 non-null  str    
 3   path            33600 non-null  str    
 4   x               33400 non-null  float64
 5   y               33400 non-null  float64
 6   z               33400 non-null  float64
dtypes: float64(3), str(4)
memory usage: 1.8 MB


Los valores nulos observados en las coordenadas x, y y z proceden del archivo XML de origen y se conservan para mantener la integridad de la información.

In [49]:
print("Puntos únicos:", puntos_bicicleta_iniciales["point_id"].nunique())
print("Duplicados:", puntos_bicicleta_iniciales.duplicated().sum())

print("\nValores nulos:")
display(puntos_bicicleta_iniciales.isnull().sum())

print("\nRutas con coordenadas nulas:")
display(
    puntos_bicicleta_iniciales.loc[
        puntos_bicicleta_iniciales[["x", "y", "z"]]
        .isnull()
        .any(axis=1),
        "path"
    ].value_counts()
)

print("\nDimensiones finales:", puntos_bicicleta_iniciales.shape)
puntos_bicicleta_iniciales.head()

Puntos únicos: 33600
Duplicados: 0

Valores nulos:


point_id            0
measurement_id      0
session_id          0
path                0
x                 200
y                 200
z                 200
dtype: int64


Rutas con coordenadas nulas:


path
front_tire/location    100
aero_pad/contour       100
Name: count, dtype: int64


Dimensiones finales: (33600, 7)


,point_id,measurement_id,session_id,path,x,y,z
0,PNT_INI_000001,MEA_INI_001,SES_001,steerer/location/xyz,-370.878,-144.557,-2300.67
1,PNT_INI_000002,MEA_INI_001,SES_001,steerer/location/xyz,-370.906,-144.478,-2300.44
2,PNT_INI_000003,MEA_INI_001,SES_001,steerer/location/xyz,-371.092,-144.580,-2300.78
3,PNT_INI_000004,MEA_INI_001,SES_001,steerer/location/xyz,-370.985,-144.449,-2300.33
4,PNT_INI_000005,MEA_INI_001,SES_001,steerer/location/xyz,-370.885,-144.338,-2299.84


### Transformación de los puntos finales de la bicicleta

Se aplica el mismo proceso utilizado para los puntos iniciales: creación de `point_id`, separación de `value` en las coordenadas `x`, `y` y `z`, y eliminación de las columnas auxiliares.


In [50]:
puntos_bicicleta_finales = df_puntos_bicicleta_finales.copy()

puntos_bicicleta_finales.insert(
    0,
    "point_id",
    [f"PNT_FIN_{i:06d}" for i in range(1, len(puntos_bicicleta_finales) + 1)]
)

puntos_bicicleta_finales[["x", "y", "z"]] = (
    puntos_bicicleta_finales["value"]
    .str.split(expand=True)
    .astype(float)
)

puntos_bicicleta_finales = puntos_bicicleta_finales.drop(
    columns=["phase", "value"]
)

puntos_bicicleta_finales.info()

display(puntos_bicicleta_finales.head())

<class 'pandas.DataFrame'>
RangeIndex: 31800 entries, 0 to 31799
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   point_id        31800 non-null  str    
 1   measurement_id  31800 non-null  str    
 2   session_id      31800 non-null  str    
 3   path            31800 non-null  str    
 4   x               31600 non-null  float64
 5   y               31600 non-null  float64
 6   z               31600 non-null  float64
dtypes: float64(3), str(4)
memory usage: 1.7 MB


,point_id,measurement_id,session_id,path,x,y,z
0,PNT_FIN_000001,MEA_FIN_001,SES_001,steerer/location/xyz,-370.681,-144.277,-2298.62
1,PNT_FIN_000002,MEA_FIN_001,SES_001,steerer/location/xyz,-370.684,-144.202,-2298.92
2,PNT_FIN_000003,MEA_FIN_001,SES_001,steerer/location/xyz,-370.902,-144.143,-2299.27
3,PNT_FIN_000004,MEA_FIN_001,SES_001,steerer/location/xyz,-371.134,-144.145,-2299.51
4,PNT_FIN_000005,MEA_FIN_001,SES_001,steerer/location/xyz,-371.199,-144.204,-2299.25


In [51]:
print("Puntos únicos:",puntos_bicicleta_finales["point_id"].nunique())

print("Duplicados:",puntos_bicicleta_finales.duplicated().sum())

print("\nValores nulos:")
display(puntos_bicicleta_finales.isnull().sum())

print("\nRutas con coordenadas nulas:")
display( puntos_bicicleta_finales.loc[ puntos_bicicleta_finales[["x", "y", "z"]].isnull().any(axis=1),"path"].value_counts())

print("\nDimensiones finales:",puntos_bicicleta_finales.shape)

puntos_bicicleta_finales.head()

Puntos únicos: 31800
Duplicados: 0

Valores nulos:


point_id            0
measurement_id      0
session_id          0
path                0
x                 200
y                 200
z                 200
dtype: int64


Rutas con coordenadas nulas:


path
front_tire/location    100
aero_pad/contour       100
Name: count, dtype: int64


Dimensiones finales: (31800, 7)


,point_id,measurement_id,session_id,path,x,y,z
0,PNT_FIN_000001,MEA_FIN_001,SES_001,steerer/location/xyz,-370.681,-144.277,-2298.62
1,PNT_FIN_000002,MEA_FIN_001,SES_001,steerer/location/xyz,-370.684,-144.202,-2298.92
2,PNT_FIN_000003,MEA_FIN_001,SES_001,steerer/location/xyz,-370.902,-144.143,-2299.27
3,PNT_FIN_000004,MEA_FIN_001,SES_001,steerer/location/xyz,-371.134,-144.145,-2299.51
4,PNT_FIN_000005,MEA_FIN_001,SES_001,steerer/location/xyz,-371.199,-144.204,-2299.25


## Unificación de las tablas de mediciones y puntos

Para evitar duplicar estructuras con las mismas variables, las mediciones iniciales y finales se integran en una única tabla `MEDICION`. Del mismo modo, los puntos iniciales y finales se integran en `PUNTO_BICICLETA`.

La columna `measurement_type` permite distinguir entre los registros `INICIAL` y `FINAL`.


In [52]:
medicion_inicial["measurement_type"] = "INICIAL"
medicion_final["measurement_type"] = "FINAL"

medicion = pd.concat(
    [medicion_inicial, medicion_final],
    ignore_index=True
)

print("Registros de medición:", len(medicion))
print(medicion["measurement_type"].value_counts())
display(medicion.head())


Registros de medición: 200
measurement_type
INICIAL    100
FINAL      100
Name: count, dtype: int64


,measurement_id,session_id,front_wheel,rear_wheel,crank_length,armPadSpacerHeight,stem_height,stem_length,stem_angle,measurement_type
0,MEA_INI_001,SES_001,694.63,697.83,172.5,52.25,362.39,120.0,-6.0,INICIAL
1,MEA_INI_002,SES_002,701.33,696.57,170.0,43.13,357.22,90.0,-8.0,INICIAL
2,MEA_INI_003,SES_003,698.93,697.13,170.0,48.23,325.71,100.0,-6.0,INICIAL
3,MEA_INI_004,SES_004,701.14,695.46,167.5,46.88,379.15,70.0,-8.0,INICIAL
4,MEA_INI_005,SES_005,700.50,692.68,170.0,59.99,352.91,90.0,-8.0,INICIAL


In [53]:
medicion.shape

(200, 10)

In [54]:
puntos_bicicleta_iniciales["measurement_type"] = "INICIAL"
puntos_bicicleta_finales["measurement_type"] = "FINAL"

punto_bicicleta = pd.concat(
    [puntos_bicicleta_iniciales, puntos_bicicleta_finales],
    ignore_index=True
)

punto_bicicleta = punto_bicicleta.drop(
    columns=["session_id"]
)

print("Registros de puntos de bicicleta:", len(punto_bicicleta))
print(punto_bicicleta["measurement_type"].value_counts())
print(punto_bicicleta.columns.tolist())
display(punto_bicicleta.head())


Registros de puntos de bicicleta: 65400
measurement_type
INICIAL    33600
FINAL      31800
Name: count, dtype: int64
['point_id', 'measurement_id', 'path', 'x', 'y', 'z', 'measurement_type']


,point_id,measurement_id,path,x,y,z,measurement_type
0,PNT_INI_000001,MEA_INI_001,steerer/location/xyz,-370.878,-144.557,-2300.67,INICIAL
1,PNT_INI_000002,MEA_INI_001,steerer/location/xyz,-370.906,-144.478,-2300.44,INICIAL
2,PNT_INI_000003,MEA_INI_001,steerer/location/xyz,-371.092,-144.580,-2300.78,INICIAL
3,PNT_INI_000004,MEA_INI_001,steerer/location/xyz,-370.985,-144.449,-2300.33,INICIAL
4,PNT_INI_000005,MEA_INI_001,steerer/location/xyz,-370.885,-144.338,-2299.84,INICIAL


In [55]:
punto_bicicleta.shape

(65400, 7)

### Tablas finales del proceso ETL

| Tabla | Registros esperados |
|---|---:|
| `CICLISTA` | 100 |
| `PEDAL` | 12 |
| `SILLIN` | 10 |
| `BICICLETA` | 100 |
| `SESION` | 100 |
| `MEDICION` | 200 |
| `PUNTO_BICICLETA` | 65.400 |


# Parte IV. Creación de la base de datos y conexión Python–MySQL

In [56]:
HOST = "127.0.0.1"
PORT = 3306
USER = "root"
DATABASE = "biomecanica_ciclista"

PASSWORD = getpass.getpass(
    "Introduce la contraseña de MySQL: "
)

print("Conectando con MySQL...")

conexion = mysql.connector.connect(
    host=HOST,
    port=PORT,
    user=USER,
    password=PASSWORD,
    database=DATABASE,
    connection_timeout=10
)

cursor = conexion.cursor()

cursor.execute("SELECT DATABASE()")
base_datos_activa = cursor.fetchone()[0]

print("Conexión con MySQL establecida.")
print("Base de datos activa:", base_datos_activa)


Conectando con MySQL...
Conexión con MySQL establecida.
Base de datos activa: biomecanica_ciclista


Creamos las tablas y conexiones con MySQL

In [61]:
cursor.execute("SET FOREIGN_KEY_CHECKS = 0")

tablas = [
    "PUNTO_BICICLETA",
    "MEDICION",
    "SESION",
    "BICICLETA",
    "SILLIN",
    "PEDAL",
    "CICLISTA"
]

for tabla in tablas:
    cursor.execute(f"DROP TABLE IF EXISTS {tabla}")

cursor.execute("SET FOREIGN_KEY_CHECKS = 1")
conexion.commit()

print("Tablas anteriores eliminadas correctamente.")


Tablas anteriores eliminadas correctamente.


In [62]:
# 1. CICLISTA
cursor.execute("""
CREATE TABLE CICLISTA (
    cyclist_id VARCHAR(20) NOT NULL,
    lastname VARCHAR(100),
    firstname VARCHAR(100),
    gender VARCHAR(20),
    birthday DATE,
    ridingstyle VARCHAR(100),
    goals TEXT,
    injuries TEXT,
    training VARCHAR(100),
    PRIMARY KEY (cyclist_id)
)
""")

# 2. PEDAL
cursor.execute("""
CREATE TABLE PEDAL (
    pedal_id VARCHAR(20) NOT NULL,
    make VARCHAR(100) NOT NULL,
    model VARCHAR(100) NOT NULL,
    PRIMARY KEY (pedal_id),
    CONSTRAINT uq_pedal_make_model UNIQUE (make, model)
)
""")

# 3. SILLIN
cursor.execute("""
CREATE TABLE SILLIN (
    saddle_id VARCHAR(20) NOT NULL,
    make VARCHAR(100) NOT NULL,
    model VARCHAR(100) NOT NULL,
    PRIMARY KEY (saddle_id),
    CONSTRAINT uq_saddle_make_model UNIQUE (make, model)
)
""")

# 4. BICICLETA
cursor.execute("""
CREATE TABLE BICICLETA (
    bike_id VARCHAR(20) NOT NULL,
    cyclist_id VARCHAR(20) NOT NULL,
    pedal_id VARCHAR(20) NOT NULL,
    saddle_id VARCHAR(20) NOT NULL,
    make VARCHAR(100),
    model VARCHAR(100),
    year INT,
    size VARCHAR(20),
    class VARCHAR(100),
    isFitBike BOOLEAN,
    PRIMARY KEY (bike_id),
    CONSTRAINT fk_bike_cyclist FOREIGN KEY (cyclist_id)
        REFERENCES CICLISTA(cyclist_id)
        ON UPDATE CASCADE ON DELETE CASCADE,
    CONSTRAINT fk_bike_pedal FOREIGN KEY (pedal_id)
        REFERENCES PEDAL(pedal_id)
        ON UPDATE CASCADE ON DELETE RESTRICT,
    CONSTRAINT fk_bike_saddle FOREIGN KEY (saddle_id)
        REFERENCES SILLIN(saddle_id)
        ON UPDATE CASCADE ON DELETE RESTRICT
)
""")

# 5. SESION
cursor.execute("""
CREATE TABLE SESION (
    session_id VARCHAR(20) NOT NULL,
    cyclist_id VARCHAR(20) NOT NULL,
    bike_id VARCHAR(20) NOT NULL,
    title VARCHAR(200),
    startTime DATETIME,
    PRIMARY KEY (session_id),
    CONSTRAINT fk_session_cyclist FOREIGN KEY (cyclist_id)
        REFERENCES CICLISTA(cyclist_id)
        ON UPDATE CASCADE ON DELETE CASCADE,
    CONSTRAINT fk_session_bike FOREIGN KEY (bike_id)
        REFERENCES BICICLETA(bike_id)
        ON UPDATE CASCADE ON DELETE CASCADE
)
""")

# 6. MEDICION
cursor.execute("""
CREATE TABLE MEDICION (
    measurement_id VARCHAR(20) NOT NULL,
    session_id VARCHAR(20) NOT NULL,
    front_wheel DECIMAL(12, 3),
    rear_wheel DECIMAL(12, 3),
    crank_length DECIMAL(12, 3),
    armPadSpacerHeight DECIMAL(12, 3),
    stem_height DECIMAL(12, 3),
    stem_length DECIMAL(12, 3),
    stem_angle DECIMAL(12, 3),
    measurement_type ENUM('INICIAL', 'FINAL') NOT NULL,
    PRIMARY KEY (measurement_id),
    CONSTRAINT uq_measurement_session_type
        UNIQUE (session_id, measurement_type),
    CONSTRAINT fk_measurement_session FOREIGN KEY (session_id)
        REFERENCES SESION(session_id)
        ON UPDATE CASCADE ON DELETE CASCADE
)
""")

# 7. PUNTO_BICICLETA
cursor.execute("""
CREATE TABLE PUNTO_BICICLETA (
    point_id VARCHAR(25) NOT NULL,
    measurement_id VARCHAR(20) NOT NULL,
    path VARCHAR(150),
    x DECIMAL(12, 3),
    y DECIMAL(12, 3),
    z DECIMAL(12, 3),
    measurement_type ENUM('INICIAL', 'FINAL') NOT NULL,
    PRIMARY KEY (point_id),
    CONSTRAINT fk_bike_point_measurement FOREIGN KEY (measurement_id)
        REFERENCES MEDICION(measurement_id)
        ON UPDATE CASCADE ON DELETE CASCADE
)
""")

conexion.commit()

print("Las siete tablas se han creado correctamente.")

Las siete tablas se han creado correctamente.


In [63]:
cursor.execute("SHOW TABLES")

tablas_creadas = [fila[0] for fila in cursor.fetchall()]

print(tablas_creadas)

['bicicleta', 'ciclista', 'medicion', 'pedal', 'punto_bicicleta', 'sesion', 'sillin']


## Carga de los DataFrames en MySQL

La función comprueba que las columnas del DataFrame coincidan con las columnas de la tabla. Después convierte valores de Pandas y NumPy a tipos compatibles con MySQL y realiza la inserción mediante `executemany`.


In [64]:
def convertir_valor_mysql(valor):
    """Convierte valores de Pandas y NumPy a tipos compatibles con MySQL."""

    if pd.isna(valor):
        return None

    if isinstance(valor, pd.Timestamp):
        return valor.to_pydatetime()

    if isinstance(valor, np.generic):
        return valor.item()

    return valor


def cargar_dataframe_mysql(dataframe, nombre_tabla, conexion, cursor):
    """Valida e inserta un DataFrame completo en una tabla MySQL."""

    cursor.execute(
        f"SHOW COLUMNS FROM `{nombre_tabla}`"
    )

    columnas_tabla = [
        fila[0]
        for fila in cursor.fetchall()
    ]

    columnas_dataframe = dataframe.columns.tolist()

    faltantes_dataframe = [
        columna
        for columna in columnas_tabla
        if columna not in columnas_dataframe
    ]

    sobrantes_dataframe = [
        columna
        for columna in columnas_dataframe
        if columna not in columnas_tabla
    ]

    if faltantes_dataframe or sobrantes_dataframe:
        raise ValueError(
            f"Las columnas de {nombre_tabla} no coinciden. "
            f"Faltan en el DataFrame: {faltantes_dataframe}. "
            f"Sobran en el DataFrame: {sobrantes_dataframe}."
        )

    dataframe_ordenado = dataframe[
        columnas_tabla
    ]

    columnas_sql = ", ".join(
        f"`{columna}`"
        for columna in columnas_tabla
    )

    marcadores = ", ".join(
        ["%s"] * len(columnas_tabla)
    )

    consulta = (
        f"INSERT INTO `{nombre_tabla}` "
        f"({columnas_sql}) "
        f"VALUES ({marcadores})"
    )

    registros = [
        tuple(
            convertir_valor_mysql(valor)
            for valor in fila
        )
        for fila in dataframe_ordenado.itertuples(
            index=False,
            name=None
        )
    ]

    try:
        cursor.executemany(
            consulta,
            registros
        )

        conexion.commit()

        print(
            f"{nombre_tabla}: "
            f"{len(registros)} registros insertados."
        )

    except Exception:
        conexion.rollback()
        raise

In [65]:
# Ajuste de la posición del sillín según la simulación definida
rutas_sillin = [
    "seat_point/location/xyz",
    "saddle/contour/xyz"
]

for sesion_id, cambio_sillin in cambio_sillin_por_sesion.items():

    # Solo corregimos las sesiones donde NO debe cambiar el sillín
    if not cambio_sillin:

        numero = sesion_id.split("_")[-1]

        medicion_inicial = f"MEA_INI_{numero}"
        medicion_final = f"MEA_FIN_{numero}"

        mascara_inicial = (
            (punto_bicicleta["measurement_id"] == medicion_inicial)
            & (punto_bicicleta["path"] == "seat_point/location/xyz")
        )

        mascara_final = (
            (punto_bicicleta["measurement_id"] == medicion_final)
            & (punto_bicicleta["path"] == "seat_point/location/xyz")
        )

        centroide_inicial = (
            punto_bicicleta.loc[
                mascara_inicial,
                ["x", "y", "z"]
            ]
            .mean()
        )

        centroide_final = (
            punto_bicicleta.loc[
                mascara_final,
                ["x", "y", "z"]
            ]
            .mean()
        )

        desplazamiento = centroide_inicial - centroide_final

        mascara_sillin_final = (
            (punto_bicicleta["measurement_id"] == medicion_final)
            & (punto_bicicleta["path"].isin(rutas_sillin))
        )

        punto_bicicleta.loc[
            mascara_sillin_final,
            ["x", "y", "z"]
        ] = (
            punto_bicicleta.loc[
                mascara_sillin_final,
                ["x", "y", "z"]
            ]
            + desplazamiento.values
        )

In [69]:
cargar_dataframe_mysql(
    ciclista,
    "CICLISTA",
    conexion,
    cursor
)

cargar_dataframe_mysql(
    df_pedal,
    "PEDAL",
    conexion,
    cursor
)

cargar_dataframe_mysql(
    df_sillin,
    "SILLIN",
    conexion,
    cursor
)

cargar_dataframe_mysql(
    bicicleta,
    "BICICLETA",
    conexion,
    cursor
)

cargar_dataframe_mysql(
    sesion,
    "SESION",
    conexion,
    cursor
)

cargar_dataframe_mysql(
    medicion,
    "MEDICION",
    conexion,
    cursor
)

cargar_dataframe_mysql(
    punto_bicicleta,
    "PUNTO_BICICLETA",
    conexion,
    cursor
)


CICLISTA: 100 registros insertados.
PEDAL: 12 registros insertados.
SILLIN: 10 registros insertados.
BICICLETA: 100 registros insertados.
SESION: 100 registros insertados.
MEDICION: 200 registros insertados.
PUNTO_BICICLETA: 65400 registros insertados.


## Validación final de la base de datos


In [57]:
tablas_validar = [
    "CICLISTA",
    "PEDAL",
    "SILLIN",
    "BICICLETA",
    "SESION",
    "MEDICION",
    "PUNTO_BICICLETA"
]

for tabla in tablas_validar:
    cursor.execute(f"SELECT COUNT(*) FROM `{tabla}`")
    total = cursor.fetchone()[0]

    print(f"{tabla:<20} {total}")


CICLISTA             100
PEDAL                12
SILLIN               10
BICICLETA            100
SESION               100
MEDICION             200
PUNTO_BICICLETA      65400


In [73]:
# Seleccionamos únicamente el punto de referencia del sillín
sillin_xyz = punto_bicicleta[
    punto_bicicleta["path"] == "seat_point/location/xyz"
].copy()

# Identificamos sesión y momento de medición
sillin_xyz["sesion_num"] = (
    sillin_xyz["measurement_id"]
    .str.extract(r"(\d+)$")[0]
)

sillin_xyz["momento"] = np.where(
    sillin_xyz["measurement_id"].str.contains("_INI_"),
    "Inicial",
    "Final"
)

# Calculamos el centroide del punto de sillín
centroides = (
    sillin_xyz
    .groupby(["sesion_num", "momento"])[["x", "y", "z"]]
    .mean()
    .unstack("momento")
)

# Distancia 3D entre posición inicial y final
centroides["distancia_mm"] = np.sqrt(
    (centroides[("x", "Final")] - centroides[("x", "Inicial")]) ** 2
    + (centroides[("y", "Final")] - centroides[("y", "Inicial")]) ** 2
    + (centroides[("z", "Final")] - centroides[("z", "Inicial")]) ** 2
)

print("Sesiones analizadas:", len(centroides))
print()

print(centroides["distancia_mm"].describe())
print()

print(
    "Sesiones con distancia > 1 mm:",
    int((centroides["distancia_mm"] > 1).sum())
)

print(
    "Porcentaje:",
    round((centroides["distancia_mm"] > 1).mean() * 100, 1),
    "%"
)

Sesiones analizadas: 100

count    1.000000e+02
mean     1.834330e+01
std      1.149668e+01
min      4.556347e-13
25%      4.556347e-13
50%      2.547680e+01
75%      2.547680e+01
max      2.547680e+01
Name: distancia_mm, dtype: float64

Sesiones con distancia > 1 mm: 72
Porcentaje: 72.0 %


## Cierre de la conexión


In [72]:
cursor.close()
conexion.close()

print("Conexión con MySQL cerrada correctamente.")


Conexión con MySQL cerrada correctamente.


La carga de los DataFrames en MySQL ha finalizado correctamente.
